# Pista C — Forma regional (vecino más cercano) + Quantile Mapping (ECO | Wind)

Sandbox de investigación pedida por Pablo (ver `avance-de-proyecto.md`, Hallazgo 21 y 22):
tres cosas en paralelo, ninguna conectada todavía a `app.py`.

1. **Selección de forma regional por vecino más cercano** entre los 4 sitios con datos reales
   (San José, Nicoya, Liberia, Finca Favorita) + validación **leave-one-out**.
2. **Acceso a ERA5 vía Copernicus CDS** — qué hace falta, sin asumir.
3. **Quantile mapping** — probar el MÉTODO (percentil a percentil, no sólo la media) hoy mismo,
   sin esperar a ERA5.

**Nota sobre entornos de ejecución:** este notebook se escribió y corrió primero en un sandbox de
Claude Code sin salida de red a `power.larc.nasa.gov`, `cds.climate.copernicus.eu` ni casi ningún
otro host externo (política de egress del entorno, Hallazgo 2) — las Partes 1 y 3 (mecánica de
quantile mapping) corren de punta a punta ahí porque usan datos ya guardados en el repo (los 4
EPW reales + el export de GWA de San José). Las celdas que necesitan internet real (Parte 2, y el
intento de Parte 3 con NASA POWER real) están marcadas explícitamente y sólo van a dar resultado
corriendo esto en **Google Colab** o cualquier entorno con internet normal.

In [1]:
import os

def _find_repo_root():
    # Busca un checkout existente del repo, mirando primero relativo al cwd
    # actual (dev sandbox, o Colab en una corrida anterior de esta misma
    # sesion) y despues la ruta estandar de Colab.
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    # Siempre forzar sync exacto con origin/main, sin importar el estado
    # previo del runtime (evita quedar pegado a una copia vieja).
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 2d9b2d9 feat(fase2): curva de excedencia por residuos -- mitiga el artefacto de Hallazgo 21 (Hallazgo 22)


/home/user/eco-wind/notebooks
Commit activo: 2d9b2d9  feat(fase2): curva de excedencia por residuos -- mitiga el artefacto de Hallazgo 21 (Hallazgo 22)  (2026-08-31 20:41:08 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — Vecino más cercano (Haversine) + validación leave-one-out

`engine/formas_regionales.py` agrega la pieza que faltaba sobre `generar_clima_gwa()`
(Hallazgo 17, ya sabía tomar prestada una forma y escalarla a otra media): la **selección** de
cuál de los 4 sitios conocidos prestar, por cercanía geográfica real, en vez de tener a San José
fijo — que es justo lo que Hallazgo 18 encontró que falla -41% a -44% en Guanacaste.

In [3]:
from engine.formas_regionales import (
    excedencia_json_desde_epw, excedencia_json_desde_epw_residual,
    cargar_formas_conocidas, vecino_mas_cercano, validar_leave_one_out, _cargar_forma_san_jose,
)
from engine.epw_real import cargar_epw_real, heatmap_json_desde_epw

# Verificación de formato: la curva derivada del EPW debe calzar con el windSpeed.json real de GWA
sj_gwa = _cargar_forma_san_jose()
print(f"windSpeed.json real de San José: {len(sj_gwa['ws_json'])} puntos, "
      f"perc {sj_gwa['ws_json'][0]['perc']}-{sj_gwa['ws_json'][-1]['perc']}")

ruta_epw_sj = os.path.join(repo, "datos_clima",
                            "CRI_AL_San.Jose-Santamaria.Intl.AP.787620_TMYx.2007-2021.epw")
df_sj_epw, _ = cargar_epw_real(ruta_epw_sj)
ws_derivado = excedencia_json_desde_epw(df_sj_epw)
print(f"Derivado del EPW de San José:    {len(ws_derivado)} puntos, "
      f"perc {ws_derivado[0]['perc']}-{ws_derivado[-1]['perc']} -- "
      f"{'OK, mismo formato' if len(ws_derivado) == len(sj_gwa['ws_json']) else 'FALLO'}")

windSpeed.json real de San José: 50 puntos, perc 2-100


Derivado del EPW de San José:    50 puntos, perc 2.0-100.0 -- OK, mismo formato


### Validación leave-one-out — sin corregir (reproduce Hallazgo 21)

Para cada uno de los 4 sitios reales: tapar su propia forma, predecir con su propia media real +
la forma del vecino más cercano de los OTROS 3, comparar contra su producción real conocida. El
escenario "siempre San José" se recalcula con el pipeline actual (post-Hallazgo 20) para comparar
en igualdad de condiciones — no se reciclan los números viejos de Hallazgo 18 sin más.

In [4]:
resultado_sin_corregir = validar_leave_one_out(usar_residuo=False)
resultado_sin_corregir

,sitio,clave,media_real_m_s,kwh_real,donante_nuevo,distancia_km,kwh_nuevo,error_nuevo_pct,kwh_viejo_san_jose,error_viejo_pct
0,San José (Aeropuerto Juan Santamaría),san_jose,3.669,156.439,"Nicoya A.P. (Guanacaste, Pacífico seco)",137.458,595.916,280.926,NaN,NaN
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",nicoya,2.091,52.400,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,50.321,122.698,134.159,30.672,-41.464
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,liberia,3.629,291.487,"Nicoya A.P. (Guanacaste, Pacífico seco)",50.321,625.736,114.670,164.260,-43.648
3,"Finca Favorita (Limón, Caribe)",finca_favorita,1.413,7.439,San José (Aeropuerto Juan Santamaría),178.604,8.864,19.159,8.864,19.159


**Resultado real, no asumido:** el vecino más cercano da error MUCHO PEOR que siempre-San-José
para los 3 sitios donde hay una alternativa real (+114% a +281%), no mejor como se esperaba para
Nicoya↔Liberia (misma zona, Guanacaste). Finca Favorita no tiene una segunda opción real en su
zona (Caribe) — su vecino más cercano de los otros 3 es San José, así que el resultado es idéntico
al viejo por construcción (no se esconde: es el único de los 4 casos sin alternativa real).

Antes de descartar la idea, se investigó la causa con una prueba de **self-reconstrucción**:
reconstruir la forma de un sitio a partir de SU PROPIA curva+heatmap derivados de EPW, sin pedir
prestado a nadie.

In [5]:
from engine.simulador_pista_a import generar_clima_gwa

def epf(ws):
    # "Factor de patrón de energía": E[v^3]/media^3 -- lo que de verdad pesa
    # en una ley de potencia cúbica P ~ v^3, no la media sola.
    m = ws.mean()
    return float(np.mean(ws ** 3) / m ** 3)

formas = cargar_formas_conocidas()
filas_diag = []

for clave in ["nicoya", "liberia", "finca_favorita"]:
    sitio = formas[clave]
    df_real = sitio["df_real"]
    df_recon, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
    filas_diag.append(dict(sitio=sitio["nombre"], forma="EPW-derivada (self-recon)",
                            epf_real=epf(df_real["WS10M"].values),
                            epf_reconstruido=epf(df_recon["WS10M"].values)))

# San José: comparar además contra su forma NATIVA de GWA (panel web, no derivada de EPW)
df_recon_nativo, _ = generar_clima_gwa(formas["san_jose"]["ws_json"], formas["san_jose"]["hm_json"])
hm_sj_epw = heatmap_json_desde_epw(df_sj_epw)
ws_sj_epw_deriv = excedencia_json_desde_epw(df_sj_epw)
df_recon_epwderiv, _ = generar_clima_gwa(ws_sj_epw_deriv, hm_sj_epw)
filas_diag.append(dict(sitio="San José", forma="NATIVA de GWA (panel web)",
                        epf_real=epf(df_sj_epw["WS10M"].values),
                        epf_reconstruido=epf(df_recon_nativo["WS10M"].values)))
filas_diag.append(dict(sitio="San José", forma="EPW-derivada (self-recon)",
                        epf_real=epf(df_sj_epw["WS10M"].values),
                        epf_reconstruido=epf(df_recon_epwderiv["WS10M"].values)))

diag = pd.DataFrame(filas_diag)
diag["inflacion_pct"] = (diag["epf_reconstruido"] / diag["epf_real"] - 1) * 100
diag

,sitio,forma,epf_real,epf_reconstruido,inflacion_pct
0,"Nicoya A.P. (Guanacaste, Pacífico seco)",EPW-derivada (self-recon),3.251,6.715,106.562
1,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,EPW-derivada (self-recon),3.410,6.990,104.981
2,"Finca Favorita (Limón, Caribe)",EPW-derivada (self-recon),1.617,1.851,14.461
3,San José,NATIVA de GWA (panel web),2.263,1.892,-16.421
4,San José,EPW-derivada (self-recon),2.263,3.414,50.854


**Diagnóstico confirmado:** `generar_clima_gwa()` dibuja un percentil aleatorio independiente por
hora desde la curva de excedencia MARGINAL (que ya contiene todo el desvío del año, incluida la
variación diurna/estacional) y lo multiplica por un índice de heatmap mes×hora APARTE — reinyecta
esa misma variación una segunda vez. `E[v³]/media³` sale ~2x inflado en Nicoya/Liberia cuando la
forma es EPW-derivada; con la forma NATIVA de GWA (San José) el efecto es chico y hasta va al
revés (subestima). No es un problema del concepto "vecino más cercano" -- es un artefacto de cómo
se reconstruye la serie horaria cuando curva y heatmap vienen ambos del mismo EPW crudo.

Separado de la mecánica rota, así se ve el CONCEPTO puro (`E[v³]/media³` real, sin ninguna
reconstrucción sintética de por medio):

In [6]:
ws_reales = {"san_jose": df_sj_epw["WS10M"].values}
for clave in ["nicoya", "liberia", "finca_favorita"]:
    ws_reales[clave] = formas[clave]["df_real"]["WS10M"].values

epf_real = {k: epf(v) for k, v in ws_reales.items()}
pares = [("nicoya", "liberia"), ("nicoya", "san_jose"), ("liberia", "san_jose"),
         ("finca_favorita", "san_jose"), ("finca_favorita", "liberia")]
pd.DataFrame([dict(par=f"{a} vs {b}", diferencia_epf_pct=abs(epf_real[a]-epf_real[b])/epf_real[b]*100)
              for a, b in pares])

,par,diferencia_epf_pct
0,nicoya vs liberia,4.681
1,nicoya vs san_jose,43.630
2,liberia vs san_jose,50.683
3,finca_favorita vs san_jose,28.550
4,finca_favorita vs liberia,52.583


Nicoya y Liberia difieren sólo ~4.7% entre sí en `E[v³]/media³` real, contra ~44-51% frente a San
José — confirma que sitios de la misma zona climática SÍ tienen forma real parecida. La idea de
fondo está bien fundada; el problema es sólo cómo se reconstruye la serie sintética.

### Mitigación (Hallazgo 22): curva de excedencia por residuos

`excedencia_json_desde_epw_residual()` construye la curva dividiendo `v(t)` entre el factor de
heatmap de su propio mes×hora ANTES de armar los percentiles — así `generar_clima_gwa()` sólo
aplica el patrón diurno/estacional una vez (al multiplicar por el heatmap), no dos.

In [7]:
filas_res = []
for clave in ["nicoya", "liberia", "finca_favorita"]:
    sitio = formas[clave]
    df_real = sitio["df_real"]
    hm = sitio["hm_json"]
    ws_residual_json = excedencia_json_desde_epw_residual(df_real, hm)
    df_recon_res, _ = generar_clima_gwa(ws_residual_json, hm)
    filas_res.append(dict(sitio=sitio["nombre"],
                           epf_real=epf(df_real["WS10M"].values),
                           epf_reconstruido_residuo=epf(df_recon_res["WS10M"].values)))

diag_res = pd.DataFrame(filas_res)
diag_res["inflacion_pct"] = (diag_res["epf_reconstruido_residuo"] / diag_res["epf_real"] - 1) * 100
diag_res

,sitio,epf_real,epf_reconstruido_residuo,inflacion_pct
0,"Nicoya A.P. (Guanacaste, Pacífico seco)",3.251,3.711,14.159
1,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,3.410,4.454,30.601
2,"Finca Favorita (Limón, Caribe)",1.617,1.736,7.343


La inflación baja de ~105% a ~14-30% — mejora real y grande, **no un arreglo completo** (queda
inflación residual, probablemente porque dividir por un promedio de 288 casillas mes×hora es una
manera gruesa de quitar la estacionalidad; la varianza, no sólo la media, también podría cambiar
por mes/hora). Con esta corrección, la validación leave-one-out completa:

In [8]:
resultado_con_residuo = validar_leave_one_out(usar_residuo=True)
resultado_con_residuo

,sitio,clave,media_real_m_s,kwh_real,donante_nuevo,distancia_km,kwh_nuevo,error_nuevo_pct,kwh_viejo_san_jose,error_viejo_pct
0,San José (Aeropuerto Juan Santamaría),san_jose,3.669,156.439,"Nicoya A.P. (Guanacaste, Pacífico seco)",137.458,321.729,105.658,NaN,NaN
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",nicoya,2.091,52.400,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,50.321,77.361,47.637,30.672,-41.464
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,liberia,3.629,291.487,"Nicoya A.P. (Guanacaste, Pacífico seco)",50.321,337.818,15.894,164.260,-43.648
3,"Finca Favorita (Limón, Caribe)",finca_favorita,1.413,7.439,San José (Aeropuerto Juan Santamaría),178.604,8.864,19.159,8.864,19.159


**Lectura honesta, sin forzarla:** Liberia ya tiene un caso claro y real donde el vecino más
cercano gana — error nuevo +15.9% contra -43.7% del viejo (siempre San José), casi 3x mejor en
magnitud. Es la primera confirmación limpia de que prestar de la misma zona climática ayuda.
Nicoya mejora mucho en magnitud (+134%→+48%) pero queda en el mismo orden que el error viejo
(-41.5%) — ya no es claramente peor, pero tampoco es todavía una victoria clara. San José, sin un
vecino real de su propia zona entre los otros 3, se sigue prediciendo mal (+105.7%) — esperable,
no es una falla del método. Sigue sin conectarse a `app.py`: falta cerrar la inflación residual y,
sobre todo, tener más de 4 sitios reales para que esto deje de depender de un solo par
Nicoya-Liberia.

## Parte 2 — Acceso a ERA5 vía Copernicus CDS (investigado, no implementado)

Investigado qué hace falta para acceder a ERA5 (no asumido):

- **Registro gratuito** — nombre, email, país, sector. Sin mención de nivel pago para el acceso
  estándar.
- Una vez registrado, un **Personal Access Token** aparece en la página de perfil de la cuenta.
- Se guarda en `$HOME/.cdsapirc`:
  ```
  url: https://cds.climate.copernicus.eu/api
  key: <PERSONAL-ACCESS-TOKEN>
  ```
- Acceso programático vía el paquete oficial `cdsapi` (`pip install cdsapi`).
- Hay que aceptar los **Términos y Condiciones de cada dataset** (ERA5 incluido) antes de poder
  descargarlo — un paso aparte del registro general.

Como ERA5 (~31km) es más fino que NASA POWER (~50-60km) pero usa el MISMO método de corrección
(quantile mapping, Parte 3) que ya se probó y funciona, perseguir el registro tiene sentido cuando
haya un sitio concreto que lo necesite — no es bloqueante para seguir probando el método con NASA
POWER, que ya es accesible en producción (sólo bloqueado en el sandbox de desarrollo).

Fuentes: [CDSAPI setup - Climate Data Store](https://cds.climate.copernicus.eu/how-to-api),
[ecmwf/cdsapi (GitHub)](https://github.com/ecmwf/cdsapi).

**Celda opcional — sólo tiene sentido correrla con internet real (Colab):** prueba de
conectividad simple, para confirmar que ya no está bloqueado como en el sandbox de desarrollo.

In [9]:
import requests

try:
    resp = requests.get("https://cds.climate.copernicus.eu/how-to-api", timeout=15)
    print(f"OK -- {resp.status_code}, el host responde desde este entorno.")
except Exception as exc:
    print(f"Sigue bloqueado (o no hay red) desde este entorno: {exc!r}")
    print("Confirmado igual en el sandbox de desarrollo de Claude Code (Hallazgo 2) -- "
          "correr esta celda en Colab para ver si cambia.")

Sigue bloqueado (o no hay red) desde este entorno: ProxyError(MaxRetryError("HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /how-to-api (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
Confirmado igual en el sandbox de desarrollo de Claude Code (Hallazgo 2) -- correr esta celda en Colab para ver si cambia.


## Parte 3 — Quantile mapping: mecánica probada (`engine/quantile_mapping.py`)

**Limitación real confirmada antes de probar nada:** la corrida real de NASA POWER de Hallazgo 1
(San José, 1.30 m/s vs 4.03 m/s del EPW real) se hizo en Colab y su serie horaria cruda nunca se
guardó en el repo — sólo sobrevive la media y el kWh derivado. No se puede probar la corrección
contra NASA POWER real todavía con lo que hay guardado; se prueba la MECÁNICA del método con un
sesgo sintético controlado sobre el EPW real de San José (magnitud real de Hallazgo 1 + una
compresión de forma sintética, marcada como tal, para imitar que NASA POWER promedia una celda de
~50-60km).

Diseño anti-tautológico: la tabla de mapeo se ajusta SÓLO con enero-junio; la comparación se hace
en julio-diciembre, que el ajuste nunca vio.

In [10]:
from engine.quantile_mapping import (
    probar_quantile_mapping_sintetico, FACTOR_MAGNITUD_NASA_POWER_SAN_JOSE,
    ajustar_quantile_mapping, aplicar_quantile_mapping,
)

print(f"Factor de magnitud usado (real, Hallazgo 1): {FACTOR_MAGNITUD_NASA_POWER_SAN_JOSE:.4f} "
      f"(= 1.30 / 4.03 m/s)")
resultado_qm_sintetico = probar_quantile_mapping_sintetico(ruta_epw_sj)
resultado_qm_sintetico

Factor de magnitud usado (real, Hallazgo 1): 0.3226 (= 1.30 / 4.03 m/s)


,version,media_m_s,cv,epf,kwh_periodo_prueba,error_kwh_pct
0,"VERDAD (EPW real, jul-dic, nunca vista por el ...",3.432,0.624,2.356,79.107,0.000
1,SESGADA cruda (sin corregir),1.204,0.287,1.265,1.051,-98.672
2,"Corregida NAIVE (solo razon de medias, ajustad...",3.996,0.287,1.265,67.146,-15.120
3,Corregida QUANTILE MAPPING (percentil a percen...,3.432,0.624,2.356,79.105,-0.003


Quantile mapping recupera la media, el CV, `E[v³]/media³` y el kWh casi exactos, FUERA de muestra
(nunca vio julio-diciembre durante el ajuste). La corrección naive (equivalente a lo que ya hace
`media_objetivo` en `generar_clima_gwa()`) sólo arregla la media — se queda en -15% de error
porque el sesgo sintético también achata la forma. Esto prueba que el método está bien
implementado y generaliza fuera de muestra bajo un sesgo ESTACIONARIO (mismo factor todo el año)
— no todavía que el sesgo real de NASA POWER se comporte así de limpio.

### Celda opcional — sólo da algo nuevo con internet real (Colab)

Intenta una descarga REAL de NASA POWER en la misma coordenada exacta del EPW de San José y, si
funciona, corre la MISMA prueba de quantile mapping (ajuste ene-jun, evaluación jul-dic) pero
contra el sesgo REAL de NASA POWER en vez del sintético de arriba. Si esto corre con éxito, el
resultado es nuevo y real — no está en `avance-de-proyecto.md` todavía porque no se pudo correr en
el sandbox de desarrollo.

In [11]:
import calendar

NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
LAT, LON, YEAR = 10.00342327565566, -84.20332993360161, 2023  # Aeropuerto Juan Santamaria, misma coordenada que el EPW real


def fetch_nasa_power_hourly(lat, lon, year, community="SB", parameters=("WS10M",)):
    params = {
        "parameters": ",".join(parameters), "community": community,
        "longitude": lon, "latitude": lat,
        "start": f"{year}0101", "end": f"{year}1231", "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


try:
    df_nasa = fetch_nasa_power_hourly(LAT, LON, YEAR)
    print(f"OK -- {len(df_nasa)} horas reales descargadas de NASA POWER.")

    from engine.simulador_pista_a import simular as simular_pista_a

    df_epw_sj_year, _ = cargar_epw_real(ruta_epw_sj, year=YEAR)
    ws_nasa, ws_epw = df_nasa["WS10M"], df_epw_sj_year["WS10M"]
    print(f"Media NASA POWER real: {ws_nasa.mean():.3f} m/s  |  "
          f"Media EPW real (verdad): {ws_epw.mean():.3f} m/s  |  "
          f"razon: {ws_nasa.mean() / ws_epw.mean():.3f}")

    train, test = ws_nasa.index.month <= 6, ws_nasa.index.month > 6
    cf, cv = ajustar_quantile_mapping(ws_nasa[train].values, ws_epw[train].values)
    factor_naive = ws_epw[train].mean() / ws_nasa[train].mean()

    ws_test_real = ws_epw[test]
    ws_test_cruda = ws_nasa[test]
    ws_test_naive = ws_test_cruda * factor_naive
    ws_test_qm = pd.Series(aplicar_quantile_mapping(ws_test_cruda.values, cf, cv), index=ws_test_cruda.index)

    kwh_real_periodo = simular_pista_a(pd.DataFrame({"WS10M": ws_test_real, "T2M": 22.0}), 3.0,
                                        "medium_tulip", 3, elevacion_m=921.0)["kwh_anual"]

    def resumen_real(ws, etiqueta):
        m = ws.mean()
        r = simular_pista_a(pd.DataFrame({"WS10M": ws, "T2M": 22.0}), 3.0, "medium_tulip", 3, elevacion_m=921.0)
        return dict(version=etiqueta, media_m_s=m, epf=epf(ws.values),
                    kwh_periodo_prueba=r["kwh_anual"], error_kwh_pct=(r["kwh_anual"] / kwh_real_periodo - 1) * 100)

    filas_real = [
        resumen_real(ws_test_real, "VERDAD (EPW real, jul-dic)"),
        resumen_real(ws_test_cruda, "NASA POWER cruda (sin corregir)"),
        resumen_real(ws_test_naive, "NASA POWER corregida NAIVE (razon de medias)"),
        resumen_real(ws_test_qm, "NASA POWER corregida QUANTILE MAPPING"),
    ]
    print()
    print(pd.DataFrame(filas_real).to_string(index=False))
    print()
    print(">>> ESTOS SON NUMEROS REALES, NO SINTETICOS -- si esto corrio, actualizar avance-de-proyecto.md")
    print(">>> (Hallazgo 21/22) con este resultado en vez del (o junto al) sintetico de arriba.")
except Exception as exc:
    print(f"No se pudo (mismo bloqueo de red ya documentado en Hallazgo 2, o cuota de NASA POWER, etc.): {exc!r}")
    print("Esta celda SOLO corre con datos reales en un entorno con internet normal (Colab). El resultado")
    print("sintetico de la celda anterior sigue siendo la unica validacion disponible hasta que esto corra con exito.")

No se pudo (mismo bloqueo de red ya documentado en Hallazgo 2, o cuota de NASA POWER, etc.): ProxyError(MaxRetryError("HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/hourly/point?parameters=WS10M&community=SB&longitude=-84.20332993360161&latitude=10.00342327565566&start=20230101&end=20231231&format=JSON (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
Esta celda SOLO corre con datos reales en un entorno con internet normal (Colab). El resultado
sintetico de la celda anterior sigue siendo la unica validacion disponible hasta que esto corra con exito.


## Resumen y próximos pasos

- **Parte 1:** la idea de "prestar del vecino real más cercano" está bien fundada en los datos
  reales (Nicoya y Liberia difieren sólo ~4.7% en forma real entre sí) pero un artefacto de
  `generar_clima_gwa()` la tapaba; mitigado parcialmente (curva por residuos) — Liberia ya tiene
  un caso limpio donde gana. Falta: cerrar la inflación residual (7-30%) y sumar más sitios reales.
- **Parte 2:** acceso a CDS/ERA5 mapeado, sin bloqueante de pago — pendiente de que Pablo registre
  una cuenta cuando haga falta un sitio concreto que lo justifique.
- **Parte 3:** la mecánica de quantile mapping funciona (fuera de muestra, sesgo sintético). Si la
  celda de NASA POWER real corrió en Colab, ese resultado reemplaza al sintético como la
  validación real del método.
- Nada de esto está conectado a `app.py` todavía — sigue siendo investigación, como se pidió.